# **MobileNetV2 Quantization with Fixed 8-bit Activations**

# ECE 662 — Team 9

**Owner**: Lilly Press

**Goal**: Respond to the midterm feedback by making accuracy and hardware use the same activation precision. Every experiment here keeps activations at a fixed 8 bits (A8) and varies only the weight precision (W8 or W4).

**What's new compared with notebook 02**
- Static 8-bit activation fake quantization at the input of every Conv2d and Linear layer, calibrated once on a fixed subset of the *training* split.
- Signed vs. unsigned activations detected per layer: post-ReLU6 inputs are unsigned; projection outputs and the residual stream are signed (no ReLU after the projection layer).
- W8A8 / W4A8 baselines, the 53-layer sensitivity sweep under A8, top-k for k = 10, 20, 30, 40, and the 10-seed storage-matched random baseline with 95% confidence intervals.
- Exported files for Dana (exact layer bit-widths and activation ranges) and Josie (one results table with a `config` column).

**Data use**: calibration = 1,024 training images (never used for ranking or testing); ranking = the 5,000-image validation split; reported accuracy = the full 10,000-image test set.

# 0. Settings

In [ ]:
# ---- Edit these paths if your files live somewhere else ----
CHECKPOINT_PATH = "/content/drive/MyDrive/mobilenetv2_cifar10.pth"
OUT_DIR = "/content/drive/MyDrive/ECE662_results_A8"   # all outputs are saved here

ACT_BITS = 8
CALIB_IMAGES = 1024          # fixed subset of the 45,000-image training split
CALIB_METHOD = "minmax"      # "minmax" or "percentile"
CALIB_PERCENTILE = 99.99     # used only if CALIB_METHOD == "percentile"

RUN_SWEEP = True             # 53-layer A8 sensitivity sweep (~15 min on a T4)
RUN_TOPK = True              # needs the sweep (or a saved A8 ranking)
RUN_RANDOM = True            # 10 seeds x 4 budgets = 40 test-set evaluations

K_VALUES = [0, 1, 3, 5, 10, 20, 30, 40]
RANDOM_K_VALUES = [10, 20, 30, 40]
RANDOM_SEEDS = list(range(10))
STORAGE_TOLERANCE = 0.05     # random configs must be within +/-5% of the top-k weight payload

# 1. Setup and Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import copy, hashlib, json, math, os, random, time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torchvision.models import mobilenet_v2
from torch.utils.data import DataLoader, Subset
from torch.nn.utils import fuse_conv_bn_eval

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} | device: {device}")

# 2. Confirm the Checkpoint

Post the size and SHA-256 below in the group chat. Dana's NeuroSim copy and Josie's copy should give the same hash; if not, the accuracy and hardware numbers describe different models.

In [ ]:
def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

print("Checkpoint:", CHECKPOINT_PATH)
print("Size (bytes):", os.path.getsize(CHECKPOINT_PATH))
print("SHA-256:", file_sha256(CHECKPOINT_PATH))

# 3. Load the Model and Data

In [ ]:
baseline_model = mobilenet_v2(weights=None)
baseline_model.classifier[1] = nn.Linear(baseline_model.last_channel, 10)
state_dict = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
baseline_model.load_state_dict(state_dict, strict=True)
baseline_model = baseline_model.to(device).eval()

# Same deterministic preprocessing as notebook 02 and Josie's evaluation
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

pin = device.type == "cuda"
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
train_eval_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)

test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=pin)

# Josie's split: seeded permutation, first 45,000 = training, last 5,000 = validation
split_indices = torch.randperm(len(train_eval_dataset), generator=torch.Generator().manual_seed(42))
train_indices = split_indices[:45000]
validation_indices = split_indices[45000:].tolist()

validation_loader = DataLoader(Subset(train_eval_dataset, validation_indices),
                               batch_size=256, shuffle=False, num_workers=2, pin_memory=pin)

# Calibration: a fixed subset of the TRAINING images (no augmentation), never used for ranking or testing
calib_perm = torch.randperm(len(train_indices), generator=torch.Generator().manual_seed(662))
calibration_indices = train_indices[calib_perm[:CALIB_IMAGES]].tolist()
calibration_loader = DataLoader(Subset(train_eval_dataset, calibration_indices),
                                batch_size=128, shuffle=False, num_workers=2, pin_memory=pin)

assert not set(calibration_indices) & set(validation_indices)
print(f"Test: {len(test_dataset):,} | Validation: {len(validation_indices):,} | Calibration: {len(calibration_indices):,}")

# 4. Shared Functions from Notebook 02

In [ ]:
def evaluate(model, loader):
    model.eval()
    model_device = next(model.parameters()).device
    correct = total = 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(model_device), labels.to(model_device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total


def quantize_weights(weights, bits):
    # Symmetric per-output-channel weight quantization (unchanged from notebook 02)
    if bits not in (4, 8):
        raise ValueError("bits must be 4 or 8")
    qmax = 2 ** (bits - 1) - 1
    reduce_dims = tuple(range(1, weights.ndim))
    max_abs = weights.abs().amax(dim=reduce_dims, keepdim=True)
    scale = torch.where(max_abs > 0, max_abs / qmax, torch.ones_like(max_abs))
    return torch.clamp(torch.round(weights / scale), -qmax, qmax) * scale


def eligible_layer_names(model):
    return [n for n, m in model.named_modules() if isinstance(m, (nn.Conv2d, nn.Linear))]


def quantize_model_by_layer(model, bit_config):
    layers = dict(model.named_modules())
    if set(bit_config) != set(eligible_layer_names(model)):
        raise ValueError("bit_config must contain exactly the eligible layer names.")
    q = copy.deepcopy(model).eval()
    q_layers = dict(q.named_modules())
    with torch.no_grad():
        for name, bits in bit_config.items():
            q_layers[name].weight.copy_(quantize_weights(q_layers[name].weight, bits))
    return q


def fold_batch_norm(model):
    folded = copy.deepcopy(model).eval()
    pairs = []
    def recurse(module, prefix=""):
        for name, child in list(module.named_children()):
            recurse(child, f"{prefix}.{name}" if prefix else name)
        if isinstance(module, nn.Sequential):
            children = list(module.named_children())
            for i in range(len(children) - 1):
                (cn, conv), (bn_name, bn) = children[i], children[i + 1]
                if isinstance(conv, nn.Conv2d) and isinstance(bn, nn.BatchNorm2d):
                    module._modules[cn] = fuse_conv_bn_eval(conv, bn)
                    module._modules[bn_name] = nn.Identity()
                    pairs.append(f"{prefix}.{cn}" if prefix else cn)
    recurse(folded)
    assert not any(isinstance(m, nn.BatchNorm2d) for m in folded.modules())
    return folded, pairs


folded_baseline, folded_pairs = fold_batch_norm(baseline_model)
layer_names = eligible_layer_names(folded_baseline)
folded_layers = dict(folded_baseline.named_modules())
print(f"Folded pairs: {len(folded_pairs)} | Eligible layers: {len(layer_names)}")
assert len(folded_pairs) == 52 and len(layer_names) == 53

# 5. Layer Types

Labels each of the 53 layers so results can be grouped by type and so Dana gets the layer order in a readable form.

In [ ]:
def layer_type(name, module):
    if isinstance(module, nn.Linear):
        return "classifier"
    if name == "features.0.0":
        return "first_conv"
    if name.startswith("features.18"):
        return "final_1x1"
    if module.groups > 1:
        return "depthwise"
    # Inside an inverted-residual block, the last conv is the projection.
    block = name.split(".")[1]
    convs_in_block = [n for n in layer_names if n.split(".")[1] == block and n.startswith(f"features.{block}.")]
    return "projection" if name == convs_in_block[-1] else "expansion"

layer_info = pd.DataFrame([{
    "order": i,
    "layer": n,
    "type": layer_type(n, folded_layers[n]),
    "weight_count": folded_layers[n].weight.numel(),
    "weight_shape": tuple(folded_layers[n].weight.shape),
} for i, n in enumerate(layer_names)])

print(layer_info["type"].value_counts().to_string())
assert layer_info["type"].value_counts().to_dict() == {
    "projection": 17, "depthwise": 17, "expansion": 16, "first_conv": 1, "final_1x1": 1, "classifier": 1}

# 6. Fixed 8-bit Activation Quantization

**Where**: the input of every Conv2d and Linear layer, which is what a compute-in-memory array receives.

**How**: static per-tensor ranges, calibrated once on the folded FP32 model with the calibration images and then held fixed for every configuration.
- **Unsigned** (input never negative during calibration, e.g. after ReLU6): `scale = max / (2^8 - 1)`, integer codes 0 to 255.
- **Signed** (input can be negative: the normalized image, projection outputs, the residual stream): `scale = max|x| / (2^7 - 1)`, integer codes −127 to 127.

Values outside the calibrated range are clipped. `mode="dynamic"` recomputes the range per batch instead; it is included only to compare against NeuroSim's FP mode, which quantizes each tensor by its own range.

This cell is self-contained so Josie can copy it to check W8A8 with exactly the same method.

In [ ]:
# ===================== Team 9 activation quantizer (copy this cell as-is) =====================
def fake_quantize_activation(x, lo, hi, bits=8):
    # Static per-tensor fake quantization. Unsigned if lo >= 0, otherwise signed symmetric.
    if lo >= 0:
        qmax = 2 ** bits - 1
        scale = max(hi, 1e-12) / qmax
        return torch.clamp(torch.round(x / scale), 0, qmax) * scale
    qmax = 2 ** (bits - 1) - 1
    scale = max(abs(lo), abs(hi), 1e-12) / qmax
    return torch.clamp(torch.round(x / scale), -qmax, qmax) * scale


def calibrate_activation_ranges(model, loader, names, method="minmax", percentile=99.99):
    # Record the input range of each named layer on the calibration data.
    modules = dict(model.named_modules())
    stats = {n: {"min": [], "max": [], "pct": []} for n in names}
    def make_hook(n):
        def hook(_m, inputs):
            x = inputs[0].detach()
            stats[n]["min"].append(x.min().item())
            stats[n]["max"].append(x.max().item())
            if method == "percentile":
                flat = x.flatten().float()
                if flat.numel() > 1_000_000:  # torch.quantile has a size limit
                    flat = flat[torch.randperm(flat.numel(), device=flat.device)[:1_000_000]]
                stats[n]["pct"].append(torch.quantile(flat.abs(), percentile / 100).item())
        return hook
    handles = [modules[n].register_forward_pre_hook(make_hook(n)) for n in names]
    model.eval()
    device_ = next(model.parameters()).device
    with torch.inference_mode():
        for images, _ in loader:
            model(images.to(device_))
    for h in handles:
        h.remove()
    ranges = {}
    for n in names:
        lo, hi = min(stats[n]["min"]), max(stats[n]["max"])
        if method == "percentile":
            p = float(np.mean(stats[n]["pct"]))
            lo, hi = (0.0, min(hi, p)) if lo >= 0 else (-p, p)
        ranges[n] = (float(lo), float(hi))
    return ranges


def attach_activation_quant(model, ranges, bits=8, mode="static"):
    # Add fake activation quantization at the input of every layer in `ranges`. Returns hook handles.
    modules = dict(model.named_modules())
    handles = []
    for n, (lo, hi) in ranges.items():
        signed = lo < 0
        def hook(_m, inputs, lo=lo, hi=hi, signed=signed):
            x = inputs[0]
            if mode == "dynamic":
                x_lo, x_hi = x.min().item(), x.max().item()
                x_lo = min(x_lo, -1e-12) if signed else 0.0
                return (fake_quantize_activation(x, x_lo, x_hi, bits),) + tuple(inputs[1:])
            return (fake_quantize_activation(x, lo, hi, bits),) + tuple(inputs[1:])
        handles.append(modules[n].register_forward_pre_hook(hook))
    return handles
# ==========================================================================================

In [ ]:
t0 = time.time()
act_ranges = calibrate_activation_ranges(folded_baseline, calibration_loader, layer_names,
                                         method=CALIB_METHOD, percentile=CALIB_PERCENTILE)
print(f"Calibrated {len(act_ranges)} layers in {time.time() - t0:.1f} s")

layer_info["input_signed"] = [act_ranges[n][0] < 0 for n in layer_names]
layer_info["act_min"] = [act_ranges[n][0] for n in layer_names]
layer_info["act_max"] = [act_ranges[n][1] for n in layer_names]
layer_info["act_scale"] = [
    (max(abs(lo), abs(hi)) / (2 ** (ACT_BITS - 1) - 1)) if lo < 0 else (hi / (2 ** ACT_BITS - 1))
    for lo, hi in (act_ranges[n] for n in layer_names)
]

# Sanity check: which layer types see signed inputs?
print(pd.crosstab(layer_info["type"], layer_info["input_signed"]).to_string())

**Expected pattern**: signed inputs for the first conv (normalized image), all 16 expansion layers, and the final 1×1 conv (they read the residual stream); unsigned inputs for all 17 depthwise layers, all 17 projection layers, and the classifier (they read ReLU6 outputs). If the table differs, stop and check before running anything else.

In [ ]:
expected_signed = layer_info["type"].isin(["first_conv", "expansion", "final_1x1"])
mismatch = layer_info[layer_info["input_signed"] != expected_signed]
if len(mismatch):
    display(mismatch)
    raise RuntimeError("Signed/unsigned pattern differs from the MobileNetV2 structure; check before continuing.")
print("Passed: 18 signed-input layers, 35 unsigned-input layers.")

# How large are the ranges? ReLU6 inputs should top out at or below 6.
display(layer_info.groupby("type")[["act_min", "act_max"]].agg(["min", "max"]).round(3))

# 7. Build and Evaluate One Configuration

Every configuration is built the same way: quantize a fresh copy of the folded model's weights, then attach the same fixed activation quantizers.

In [ ]:
def build_model(bit_config, act_bits=ACT_BITS, mode="static"):
    if bit_config is None:   # FP32 weights
        m = copy.deepcopy(folded_baseline).eval()
    else:
        m = quantize_model_by_layer(folded_baseline, bit_config)
    if act_bits is not None:
        attach_activation_quant(m, act_ranges, bits=act_bits, mode=mode)
    return m


def weight_payload_mb(bit_config):
    # Packed Conv2d/Linear weight payload only (decimal MB), same definition as the midterm Table 2.
    return sum(folded_layers[n].weight.numel() * b for n, b in bit_config.items()) / 8 / 1e6


def run_config(bit_config, loader, act_bits=ACT_BITS, mode="static"):
    m = build_model(bit_config, act_bits, mode)
    acc = evaluate(m, loader)
    del m
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return acc


all_w8 = {n: 8 for n in layer_names}
all_w4 = {n: 4 for n in layer_names}

# 8. Baselines (Full Test Set)

The first two rows reproduce the midterm numbers (89.82% and 89.87%). If they don't match, stop: something in the setup changed.

In [ ]:
baseline_rows = []
for label, cfg, a_bits in [
    ("FP32 (W32A32)", None, None),
    ("W8A32 (midterm W8)", all_w8, None),
    ("W4A32 (midterm W4)", all_w4, None),
    ("W32A8 (activation-only error)", None, ACT_BITS),
    ("W8A8", all_w8, ACT_BITS),
    ("W4A8", all_w4, ACT_BITS),
]:
    t0 = time.time()
    acc = run_config(cfg, test_loader, act_bits=a_bits)
    baseline_rows.append({"Configuration": label, "Test accuracy (%)": acc,
                          "Weight payload (MB)": weight_payload_mb(cfg) if cfg else weight_payload_mb({n: 32 for n in layer_names})})
    print(f"{label:32s} {acc:6.2f}%  ({time.time() - t0:.0f} s)", flush=True)

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df.round(3))

W8A8_TEST = baseline_df.loc[baseline_df["Configuration"] == "W8A8", "Test accuracy (%)"].item()
print(f"\nNew reference for the one-point target: W8A8 = {W8A8_TEST:.2f}%  ->  target >= {W8A8_TEST - 1:.2f}%")
baseline_df.to_csv(f"{OUT_DIR}/baselines_A8.csv", index=False)

**Optional comparison**: dynamic (per-batch) activation ranges, closer to NeuroSim's FP mode. Report static A8 as the main result; use this only to show whether the choice of activation range matters.

In [ ]:
for label, cfg in [("W8A8 dynamic", all_w8), ("W4A8 dynamic", all_w4)]:
    print(f"{label:16s} {run_config(cfg, test_loader, mode='dynamic'):6.2f}%")

# 9. Pilot: Top-10 with the Midterm Ranking (for the First TA Check-in)

Uses the old FP32-activation ranking so there is a top-10 A8 number to show before the new sweep finishes. Section 10 replaces it with the A8 ranking.

In [ ]:
MIDTERM_TOP10 = [
    "features.1.conv.0.0", "features.2.conv.1.0", "features.1.conv.1", "features.2.conv.2",
    "features.3.conv.1.0", "features.0.0", "features.7.conv.2", "features.4.conv.2",
    "features.14.conv.2", "features.5.conv.2",
]
assert set(MIDTERM_TOP10) <= set(layer_names)
pilot_cfg = {n: 8 if n in MIDTERM_TOP10 else 4 for n in layer_names}
pilot_acc = run_config(pilot_cfg, test_loader)
print(f"Top-10 (midterm ranking), A8: {pilot_acc:.2f}%  | midterm with FP32 activations: 79.88%  "
      f"| gap to W8A8: {W8A8_TEST - pilot_acc:.2f} pp | payload {weight_payload_mb(pilot_cfg):.4f} MB")

# 10. Layer Sensitivity Sweep under A8 (Validation Split)

Same procedure as notebook 02: start from W8A8, lower one layer to W4, measure the validation drop. Only the activations changed. Progress is saved after every layer, so a Colab disconnect doesn't lose the run.

In [ ]:
SWEEP_PATH = f"{OUT_DIR}/layer_sensitivity_A8.csv"

if RUN_SWEEP:
    w8a8_val = run_config(all_w8, validation_loader)
    print(f"W8A8 validation accuracy: {w8a8_val:.2f}%")
    rows, t0 = [], time.time()
    for i, n in enumerate(layer_names, 1):
        cfg = dict(all_w8); cfg[n] = 4
        acc = run_config(cfg, validation_loader)
        rows.append({"Layer": n, "Type": layer_info.loc[layer_info.layer == n, "type"].item(),
                     "Validation accuracy (%)": acc, "Drop from W8A8 (pp)": w8a8_val - acc})
        pd.DataFrame(rows).to_csv(f"{OUT_DIR}/layer_sensitivity_A8_progress.csv", index=False)
        print(f"[{i}/53] {n}: {acc:.2f}% | drop {w8a8_val - acc:.2f} pp", flush=True)
    sens_A8 = (pd.DataFrame(rows).sort_values("Drop from W8A8 (pp)", ascending=False, kind="stable")
               .reset_index(drop=True))
    sens_A8.insert(0, "Rank", range(1, len(sens_A8) + 1))
    sens_A8["W8A8 validation accuracy (%)"] = w8a8_val
    sens_A8.to_csv(SWEEP_PATH, index=False)
    print(f"Finished in {(time.time() - t0) / 60:.1f} min")
else:
    sens_A8 = pd.read_csv(SWEEP_PATH)

display(sens_A8.head(10))

In [ ]:
# Compare with the midterm (FP32-activation) ranking
midterm_rank = {n: i + 1 for i, n in enumerate(MIDTERM_TOP10)}
top10_A8 = sens_A8["Layer"].head(10).tolist()
print("Top-10 under A8:", top10_A8)
print("Overlap with midterm top-10:", len(set(top10_A8) & set(MIDTERM_TOP10)), "of 10")

summary_by_type = (sens_A8.groupby("Type")["Drop from W8A8 (pp)"]
                   .agg(["count", "mean", "max"]).sort_values("mean", ascending=False))
summary_by_type["in_top10"] = [sum(sens_A8.head(10)["Type"] == t) for t in summary_by_type.index]
display(summary_by_type.round(3))

# 11. Top-k Mixed Precision under A8 (Test Set)

Top-k layers from the A8 ranking stay at W8; all others use W4; activations stay at A8. k = 0 is W4A8 and k = 53 is W8A8.

In [ ]:
ranked_A8 = sens_A8["Layer"].tolist()

def topk_config(k):
    keep = set(ranked_A8[:k])
    return {n: 8 if n in keep else 4 for n in layer_names}

configs_for_dana = {"W8A8": all_w8, "W4A8": all_w4}
topk_rows = [{"config": "W8A8", "k": 53, "selection": "all", "act_bits": ACT_BITS,
              "test_accuracy": W8A8_TEST, "weight_payload_MB": weight_payload_mb(all_w8)}]

if RUN_TOPK:
    for k in K_VALUES:
        cfg = topk_config(k)
        name = "W4A8" if k == 0 else f"k_{k}"
        if k > 0:
            configs_for_dana[name] = cfg
        acc = run_config(cfg, test_loader)
        topk_rows.append({"config": name, "k": k, "selection": "sensitivity", "act_bits": ACT_BITS,
                          "test_accuracy": acc, "weight_payload_MB": weight_payload_mb(cfg)})
        print(f"{name:6s} {acc:6.2f}% | gap {W8A8_TEST - acc:6.2f} pp | {weight_payload_mb(cfg):.4f} MB", flush=True)

topk_df = pd.DataFrame(topk_rows)
topk_df["gap_to_W8A8_pp"] = W8A8_TEST - topk_df["test_accuracy"]
topk_df["within_1pp"] = topk_df["gap_to_W8A8_pp"] <= 1.0
topk_df["num_w8_layers"] = topk_df["k"]
topk_df.to_csv(f"{OUT_DIR}/mixed_precision_A8.csv", index=False)
display(topk_df.round(3))

hits = topk_df[(topk_df["selection"] == "sensitivity") & topk_df["within_1pp"]]
print("Smallest tested k within 1 pp of W8A8:", hits["k"].min() if len(hits) else "none (report as a negative result)")

# 12. Random-Protection Baseline at Matched Storage (10 Seeds)

For each k, every seed shuffles the 53 layers and keeps layers at W8 in that random order until the *extra* storage from W8 layers (over all-W4) matches the top-k configuration's extra storage within ±5%. Matching on storage, not layer count, matters because the most sensitive layers are mostly small early layers.

In [ ]:
def random_config_matched(target_mb, seed, tol=STORAGE_TOLERANCE, max_tries=5000):
    # Random layers at W8 whose EXTRA storage (over all-W4) matches the top-k extra storage within +/-tol.
    # Matching the extra storage, not the total, keeps the tolerance meaningful: for k = 10 the extra
    # W8 storage is only ~5% of the total payload.
    base_mb = weight_payload_mb(all_w4)
    extra_target = target_mb - base_mb
    extra_per_layer = {n: folded_layers[n].weight.numel() * 4 / 8 / 1e6 for n in layer_names}
    rng = random.Random(seed)
    best = None
    for attempt in range(max_tries):
        order = layer_names[:]
        rng.shuffle(order)
        chosen, extra = set(), 0.0
        for n in order:
            if extra + extra_per_layer[n] <= extra_target * (1 + tol):
                chosen.add(n); extra += extra_per_layer[n]
        err = abs(extra - extra_target) / extra_target
        if best is None or err < best[1]:
            best = ({n: 8 if n in chosen else 4 for n in layer_names}, err, attempt)
        if err <= tol:
            break
    return best   # (config, relative error of extra storage, attempts used)


T_CRIT_95_DF9 = 2.262   # two-sided 95% t critical value for 10 seeds

random_rows, random_configs = [], {}
if RUN_RANDOM:
    for k in RANDOM_K_VALUES:
        target = weight_payload_mb(topk_config(k))
        for s in RANDOM_SEEDS:
            cfg, err, attempt = random_config_matched(target, seed=1000 * k + s)
            acc = run_config(cfg, test_loader)
            name = f"rand_k{k}_s{s}"
            random_configs[name] = cfg
            random_rows.append({"config": name, "matched_k": k, "seed": s, "act_bits": ACT_BITS,
                                "num_w8_layers": sum(b == 8 for b in cfg.values()),
                                "weight_payload_MB": weight_payload_mb(cfg), "target_payload_MB": target,
                                "extra_storage_rel_err": err, "test_accuracy": acc})
            print(f"k={k} seed={s}: {acc:6.2f}% | {sum(b == 8 for b in cfg.values())} W8 layers | "
                  f"{weight_payload_mb(cfg):.4f} vs {target:.4f} MB", flush=True)
            pd.DataFrame(random_rows).to_csv(f"{OUT_DIR}/random_baseline.csv", index=False)

    random_df = pd.DataFrame(random_rows)
    summary = random_df.groupby("matched_k").agg(
        random_mean=("test_accuracy", "mean"), random_std=("test_accuracy", "std"),
        n=("test_accuracy", "size"), mean_w8_layers=("num_w8_layers", "mean"),
        mean_payload_MB=("weight_payload_MB", "mean")).reset_index()
    summary["ci95_half_width"] = T_CRIT_95_DF9 * summary["random_std"] / np.sqrt(summary["n"])
    summary["ci95_low"] = summary["random_mean"] - summary["ci95_half_width"]
    summary["ci95_high"] = summary["random_mean"] + summary["ci95_half_width"]
    summary = summary.merge(
        topk_df[topk_df.selection == "sensitivity"][["k", "test_accuracy", "weight_payload_MB"]]
        .rename(columns={"k": "matched_k", "test_accuracy": "topk_accuracy", "weight_payload_MB": "topk_payload_MB"}),
        on="matched_k")
    summary["topk_minus_random_mean_pp"] = summary["topk_accuracy"] - summary["random_mean"]
    summary["topk_above_ci95"] = summary["topk_accuracy"] > summary["ci95_high"]
    summary.to_csv(f"{OUT_DIR}/random_baseline_summary.csv", index=False)
    display(summary.round(3))

# 13. Export for Dana and Josie

In [ ]:
# Dana: exact bit-widths per layer, in network order, for every configuration to simulate
with open(f"{OUT_DIR}/mixed_precision_configs.json", "w") as f:
    json.dump({"layer_order": layer_names, "act_bits": ACT_BITS, "configs": configs_for_dana}, f, indent=2)
with open(f"{OUT_DIR}/random_baseline_configs.json", "w") as f:
    json.dump({"layer_order": layer_names, "act_bits": ACT_BITS, "configs": random_configs}, f, indent=2)

# Same assignments as a table (one row per layer, one column per config) - easier to read
bits_table = layer_info[["order", "layer", "type", "weight_count"]].copy()
for name, cfg in configs_for_dana.items():
    bits_table[name] = [cfg[n] for n in layer_names]
bits_table.to_csv(f"{OUT_DIR}/layer_bitwidths_by_config.csv", index=False)

# Dana: activation ranges and signedness, so NeuroSim can use the same activation setup
layer_info.to_csv(f"{OUT_DIR}/layer_info_and_activation_calibration.csv", index=False)

# Josie: one accuracy table to merge with storage and hardware on the `config` column
accuracy_for_merge = pd.concat([
    topk_df[["config", "k", "selection", "act_bits", "test_accuracy", "gap_to_W8A8_pp", "weight_payload_MB"]],
    pd.DataFrame(random_rows)[["config", "matched_k", "act_bits", "test_accuracy", "weight_payload_MB"]]
      .rename(columns={"matched_k": "k"}).assign(selection="random",
                                                 gap_to_W8A8_pp=lambda d: W8A8_TEST - d["test_accuracy"])
    if random_rows else None,
], ignore_index=True)
accuracy_for_merge.to_csv(f"{OUT_DIR}/accuracy_A8_for_merge.csv", index=False)

with open(f"{OUT_DIR}/run_metadata.json", "w") as f:
    json.dump({"checkpoint": CHECKPOINT_PATH, "checkpoint_sha256": file_sha256(CHECKPOINT_PATH),
               "act_bits": ACT_BITS, "calibration_images": CALIB_IMAGES, "calibration_method": CALIB_METHOD,
               "calibration_source": "training split (first 45,000 of seed-42 permutation), subset seed 662",
               "ranking_data": "validation split (last 5,000)", "reported_accuracy": "full 10,000-image test set",
               "torch": torch.__version__, "torchvision": torchvision.__version__}, f, indent=2)

print("Saved to", OUT_DIR)
for fn in sorted(os.listdir(OUT_DIR)):
    print("  ", fn)

# 14. Notes for the Report

- **What is measured vs. simulated**: accuracy is measured in PyTorch with *simulated* (fake) quantization of weights and activations, not native integer inference. Storage is an *estimate* of the packed weight payload (scales, biases, and metadata are in Josie's storage analysis).
- **Activation calibration**: static per-tensor min/max ranges from 1,024 training images, computed once on the folded FP32 model and shared by every configuration. Ranges are not recalibrated for W4 models.
- **Signedness**: 18 layers receive signed inputs (first conv, expansion layers, final 1×1); 35 receive unsigned post-ReLU6 inputs. NeuroSim's encoder assumes inputs in [−1, 1), so check with Dana how it handles the signed residual-stream inputs and ReLU6 values above 1.
- **One-point target**: now measured against W8A8 (Section 8), not the old FP32-activation W8 (89.87%).
- **Random baseline**: matched on the extra W8 weight storage (over all-W4) within ±5%, 10 seeds; 95% CI uses t(9) = 2.262. Top-k "beats random" if its accuracy is above the upper CI bound.